# Farthest-First Traversal Test on HotpotQA

Hooks into `LiteSemRAG._assign_sem_description_on_build` via `_fft_call_recorder` to capture, for every sem-node assignment:

- `token_text`, `sem_node_id`, `path_taken` (`consensus` / `split` / `no_prediction` / `no_model`)
- every text-embedding in the cluster: span text, surrounding context, and embedding vector
- the indices selected by `farthest_first_traversal` (the 10 sample points fed to the cross-encoder)
- per-sample cross-encoder predictions, and the per-embedding final assignment after consensus or d1/d2 split

Then offers a visualization that takes a `record` and a reducer (`pca` / `tsne`) and draws two scatter plots:
1. Recorded assignment (sampled points highlighted).
2. Independent cross-encoder vote per point — comparison baseline.

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Working directory: {REPO_ROOT}")

Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from text_processing import *
from utils import *

hotpot_file_candidates = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
file_path = next((str(p) for p in hotpot_file_candidates if p.exists()), str(hotpot_file_candidates[0]))
print(f"Using HotpotQA file: {file_path}")

NUM_SAMPLES = 500

# Drop in-process Wikidata lookup lru_caches so candidate banks aren't reused
# from a previous kernel run. (HotpotQA dataset on-disk cache is intentionally
# kept — it is just a parsed copy of the raw json and safe to reuse.)
import utils as _utils_module
for _attr in ("_get_wikidata_term_info_cached", "is_multi_semantic_by_wikidata"):
    _fn = getattr(_utils_module, _attr, None)
    if _fn is not None and hasattr(_fn, "cache_clear"):
        _fn.cache_clear()
        print(f"Cleared lru_cache: utils.{_attr}")

documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=NUM_SAMPLES)
print(f"Documents: {len(documents)} | Samples: {len(samples)}")


Using HotpotQA file: /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Cleared lru_cache: utils._get_wikidata_term_info_cached
Cleared lru_cache: utils.is_multi_semantic_by_wikidata
Loading cached dataset...
Loaded 4937 documents
Loaded 500 samples
Documents: 4937 | Samples: 500


In [3]:
import pickle
from collections import Counter
from datetime import datetime
from pathlib import Path

import RAG_graph

FFT_CACHE_DIR = REPO_ROOT / "cache" / "fft_traversal_records"
FFT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
USE_FFT_CACHE = True  # reuse cached fft_records by default
FFT_CACHE_PATH = None  # set to Path(...) to force a specific cache file
REBUILD_GRAPH = False  # set True only when you want to rebuild LiteSemRAG and refresh fft_records

GRAPH_BATCH_SIZE = 4
GRAPH_CONFIG = dict(
    text_embed_dim=1024,
    df_ratio=0.9,
    buffer_size=100,
    chunk_size=256,
    device="cuda",
    plot_embeds=False,
    consensus_ratio_threshold=0.9,
    min_description_candidates=3,
)

def _latest_fft_cache_path(cache_dir):
    cache_paths = sorted(cache_dir.glob("records_*.pkl"))
    return cache_paths[-1] if cache_paths else None

resolved_fft_cache_path = Path(FFT_CACHE_PATH) if FFT_CACHE_PATH else _latest_fft_cache_path(FFT_CACHE_DIR)
generated_fft_cache_path = None
loaded_from_cache = False
graph_database = None
fft_records = []

if USE_FFT_CACHE and resolved_fft_cache_path is not None and resolved_fft_cache_path.exists() and not REBUILD_GRAPH:
    with open(resolved_fft_cache_path, "rb") as f:
        fft_records = pickle.load(f)
    loaded_from_cache = True
    print(f"Loaded {len(fft_records)} FFT records from {resolved_fft_cache_path}")
else:
    def fft_recorder(payload):
        # `payload` carries everything emitted by _assign_sem_description_on_build.
        # Embeddings are already converted to numpy arrays in the snapshot.
        fft_records.append(payload)

    graph_database = RAG_graph.LiteSemRAG(**GRAPH_CONFIG)
    graph_database._fft_call_recorder = fft_recorder  # install hook before indexing
    graph_database.index_json(documents, batch_size=GRAPH_BATCH_SIZE)
    graph_database.finalize()

    generated_fft_cache_path = FFT_CACHE_DIR / f"records_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
    with open(generated_fft_cache_path, "wb") as f:
        pickle.dump(fft_records, f)
    print(f"Saved {len(fft_records)} FFT records to {generated_fft_cache_path}")

print(f"Captured {len(fft_records)} FFT call records.")
print("Path distribution:", Counter(r["path_taken"] for r in fft_records))
if loaded_from_cache and graph_database is None:
    print("Loaded cache only: graph_database is unavailable, so per-point cross-encoder validation will be skipped.")

Loaded 40 FFT records from /home/xiaoyue/LiteSemRAG/cache/fft_traversal_records/records_20260425_173600.pkl
Captured 40 FFT call records.
Path distribution: Counter({'consensus': 29, 'split': 11})
Loaded cache only: graph_database is unavailable, so per-point cross-encoder validation will be skipped.


In [4]:
current_fft_cache_path = resolved_fft_cache_path if loaded_from_cache else generated_fft_cache_path
print(f"Current FFT cache path: {current_fft_cache_path}")
print(f"Loaded from cache: {loaded_from_cache}")
print(f"Live graph available: {graph_database is not None}")

Current FFT cache path: /home/xiaoyue/LiteSemRAG/cache/fft_traversal_records/records_20260425_173600.pkl
Loaded from cache: True
Live graph available: False


In [5]:
# Inspect what got captured. Filter to records with both:
#   - enough points for a meaningful 2D plot (>=12), and
#   - more than one final assignment description (i.e. split path), to make plot 1 colorful.
import pandas as pd

rows = []
for idx, rec in enumerate(fft_records):
    final_descs = {a["assigned_description"] for a in rec.get("final_assignments", [])}
    rows.append({
        "record_idx": idx,
        "token_text": rec["token_text"],
        "sem_node_id": rec["sem_node_id"],
        "path_taken": rec["path_taken"],
        "n_points": len(rec["all_text_embeddings"]),
        "n_sampled": len(rec.get("sampled_indices", [])),
        "final_desc_count": len(final_descs),
        "prediction_top": rec.get("prediction_top_description"),
    })
df = pd.DataFrame(rows).sort_values(["path_taken", "n_points"], ascending=[True, False]).reset_index(drop=True)
df.head(40)

,record_idx,token_text,sem_node_id,path_taken,n_points,n_sampled,final_desc_count,prediction_top
0,0,song,0,consensus,475,10,1,musical composition for voice(s)
1,2,united states,4,consensus,432,10,1,country located primarily in North America
2,6,series,8,consensus,377,10,1,term in archival and library science; collecti...
3,5,time,7,consensus,354,10,1,dimension in which events can be ordered from ...
4,3,album,5,consensus,348,10,1,grouping of album releases by an artist usuall...
5,4,member,6,consensus,297,10,1,any one of the distinct objects that make up a...
6,8,single,10,consensus,255,10,1,type of music release
7,13,band,23,consensus,238,10,1,musical ensemble which performs music
8,7,team,9,consensus,186,10,1,group linked in a common purpose
9,10,city,15,consensus,182,10,1,large human settlement


In [6]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sentence_transformers import CrossEncoder

from utils import definition_to_hypothesis, extract_cross_encoder_scores


CROSS_ENCODER_MODEL_NAME = GRAPH_CONFIG.get("sem_description_model_name", "cross-encoder/nli-deberta-v3-large")
_crossencoder_model_cache = None


def _get_crossencoder_model():
    global _crossencoder_model_cache
    if _crossencoder_model_cache is None:
        _crossencoder_model_cache = CrossEncoder(CROSS_ENCODER_MODEL_NAME)
    return _crossencoder_model_cache


def _reduce_2d(embeddings, method="pca", random_state=42):
    method = method.lower()
    if method == "pca":
        return PCA(n_components=2, random_state=random_state).fit_transform(embeddings)
    if method == "tsne":
        perplexity = min(30, max(2, len(embeddings) // 3))
        return TSNE(
            n_components=2,
            random_state=random_state,
            init="random",
            learning_rate="auto",
            perplexity=perplexity,
        ).fit_transform(embeddings)
    raise ValueError(f"unsupported reducer: {method!r} (use 'pca' or 'tsne')")


def _scatter_by_label(ax, coords, labels, sampled_mask, title, all_candidate_labels=None):
    observed = sorted({lbl for lbl in labels if lbl is not None})
    if all_candidate_labels is None:
        legend_labels = observed
    else:
        legend_labels = list(all_candidate_labels)
        for lbl in observed:
            if lbl not in legend_labels:
                legend_labels.append(lbl)
    base_colors = [
        "#d62728",  # red
        "#1f77b4",  # blue
        "#2ca02c",  # green
        "#ff7f0e",  # orange
        "#9467bd",  # purple
        "#8c564b",  # brown
        "#e377c2",  # pink
        "#7f7f7f",  # gray
        "#bcbd22",  # olive
        "#17becf",  # cyan
    ]
    color_map = {
        lbl: base_colors[i % len(base_colors)]
        for i, lbl in enumerate(legend_labels)
    }
    counts = {lbl: int(sum(1 for l in labels if l == lbl)) for lbl in legend_labels}

    for lbl in legend_labels:
        mask = np.array([l == lbl for l in labels]) & ~sampled_mask
        legend_text = f"{lbl} (n={counts[lbl]})"
        if mask.any():
            ax.scatter(coords[mask, 0], coords[mask, 1], s=22, alpha=0.7,
                       color=color_map[lbl], label=legend_text)
        else:
            ax.scatter([], [], s=22, color=color_map[lbl], label=legend_text)
    none_mask = np.array([l is None for l in labels]) & ~sampled_mask
    if none_mask.any():
        ax.scatter(coords[none_mask, 0], coords[none_mask, 1], s=22, alpha=0.5,
                   color="lightgray", label=f"(unassigned) (n={int(none_mask.sum())})")
    if sampled_mask.any():
        for lbl in legend_labels:
            mask = np.array([l == lbl for l in labels]) & sampled_mask
            if mask.any():
                ax.scatter(coords[mask, 0], coords[mask, 1], s=160, marker="*",
                           edgecolor="black", linewidth=0.8, color=color_map[lbl])
        none_sampled = np.array([l is None for l in labels]) & sampled_mask
        if none_sampled.any():
            ax.scatter(coords[none_sampled, 0], coords[none_sampled, 1], s=160, marker="*",
                       edgecolor="black", linewidth=0.8, color="lightgray")
    ax.set_title(title)
    ax.set_xlabel("comp 1")
    ax.set_ylabel("comp 2")
    ax.legend(loc="best", fontsize=6, frameon=True)


def _candidate_bank_from_record(record):
    bank = []
    for cand in record.get("candidate_bank", []) or []:
        definition = str(cand.get("definition") or "").strip()
        description = cand.get("description")
        if not definition or not description:
            continue
        bank.append({
            "entity_id": cand.get("entity_id"),
            "label": cand.get("label"),
            "description": description,
            "definition": definition,
            "definition_source": cand.get("definition_source"),
            "hypothesis": definition_to_hypothesis(definition),
        })
    return bank


def _candidate_descriptions_from_record(record, graph=None):
    bank = _candidate_bank_from_record(record)
    if bank:
        return [cand["description"] for cand in bank]
    return []


def _build_cached_prompt_text(record, point):
    context_text = str(point.get("context_text") or "").strip()
    token_text = str(record.get("token_text") or "").strip()
    if not context_text or not token_text:
        return None
    return (
        f"Context: {context_text}\n"
        f"Target word: {token_text}\n\n"
        f'Question: What does "{token_text}" mean in this context?'
    )


def _crossencoder_per_point_labels(graph, record):
    candidate_bank = _candidate_bank_from_record(record)
    if not candidate_bank:
        return None

    model = _get_crossencoder_model()
    labels = []
    batch_size = min(32, len(candidate_bank))
    for point in record["all_text_embeddings"]:
        prompt_text = _build_cached_prompt_text(record, point)
        if prompt_text is None:
            labels.append(None)
            continue
        pairs = [(prompt_text, candidate["hypothesis"]) for candidate in candidate_bank]
        raw_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        scores = extract_cross_encoder_scores(raw_scores, model)
        ranked_candidates = sorted(
            [
                {
                    **candidate,
                    "score": float(score),
                }
                for candidate, score in zip(candidate_bank, scores)
            ],
            key=lambda item: item["score"],
            reverse=True,
        )
        labels.append(ranked_candidates[0]["description"] if ranked_candidates else None)
    return labels


def _normalize_label(label):
    return label if label is not None else "(unassigned)"


def _label_counts(labels):
    return Counter(_normalize_label(label) for label in labels)


def _label_deltas(recorded_labels, crossenc_labels):
    recorded_counts = _label_counts(recorded_labels)
    if crossenc_labels is None:
        return recorded_counts, None, None
    crossenc_counts = _label_counts(crossenc_labels)
    deltas = {}
    for label in sorted(set(recorded_counts) | set(crossenc_counts)):
        deltas[label] = crossenc_counts.get(label, 0) - recorded_counts.get(label, 0)
    return recorded_counts, crossenc_counts, deltas


def _safe_slug(text, max_len=80):
    slug = re.sub(r"[^A-Za-z0-9_-]+", "-", str(text)).strip("-")
    slug = slug or "record"
    return slug[:max_len]


def _build_record_report_lines(record, export_result, method):
    lines = []
    lines.append(f"method={method}")
    lines.append(f"record_idx={export_result['record_index']}")
    lines.append(f"token_text={record['token_text']}")
    lines.append(f"sem_node_id={record['sem_node_id']}")
    lines.append(f"path_taken={record['path_taken']}")
    lines.append(f"n_points={len(record['all_text_embeddings'])}")
    lines.append(f"n_sampled={len(record.get('sampled_indices', []))}")
    lines.append(f"image_path={export_result['image_path']}")
    lines.append(f"crossencoder_available={export_result['crossencoder_available']}")
    candidate_descriptions = export_result.get('candidate_descriptions', [])
    lines.append(f"candidate_count={len(candidate_descriptions)}")
    if candidate_descriptions:
        lines.append("candidate_descriptions=")
        for idx, desc in enumerate(candidate_descriptions):
            lines.append(f"  [{idx}] {desc}")
    recorded_counts = export_result['recorded_counts']
    lines.append("record_assignment_counts=")
    for label, count in sorted(recorded_counts.items()):
        lines.append(f"  {label}: {count}")
    if export_result['crossencoder_counts'] is None:
        lines.append("crossencoder_validation_counts=UNAVAILABLE")
        lines.append("crossencoder_minus_record_assignment=UNAVAILABLE")
    else:
        lines.append("crossencoder_validation_counts=")
        for label, count in sorted(export_result['crossencoder_counts'].items()):
            lines.append(f"  {label}: {count}")
        lines.append("crossencoder_minus_record_assignment=")
        for label, delta in sorted(export_result['count_deltas'].items()):
            if delta > 0:
                status = f"more_by_{delta}"
            elif delta < 0:
                status = f"less_by_{abs(delta)}"
            else:
                status = "same"
            lines.append(
                f"  {label}: delta={delta} ({status}), "
                f"recorded={recorded_counts.get(label, 0)}, "
                f"crossencoder={export_result['crossencoder_counts'].get(label, 0)}"
            )
    lines.append("")
    return lines


def visualize_fft_record(
    record,
    graph=None,
    method="pca",
    random_state=42,
    figsize=(17, 7),
    save_path=None,
    record_index=None,
):
    points = record["all_text_embeddings"]
    if len(points) < 2:
        return None
    embeds = np.stack([p["embedding"] for p in points])
    coords = _reduce_2d(embeds, method=method, random_state=random_state)

    sampled_mask = np.zeros(len(points), dtype=bool)
    for idx in record.get("sampled_indices", []):
        if 0 <= idx < len(points):
            sampled_mask[idx] = True

    recorded_labels = [None] * len(points)
    for entry in record.get("final_assignments", []):
        ti = entry["te_index"]
        if 0 <= ti < len(points):
            recorded_labels[ti] = entry["assigned_description"]

    candidate_descriptions = _candidate_descriptions_from_record(record, graph)
    crossenc_labels = _crossencoder_per_point_labels(graph, record)
    recorded_counts, crossenc_counts, count_deltas = _label_deltas(recorded_labels, crossenc_labels)

    if crossenc_labels is None:
        validation_title = (
            f"{method.upper()} | token={record['token_text']!r} | sem_node={record['sem_node_id']} | "
            f"path={record['path_taken']}\n(cross-encoder per-point vote unavailable from cached data)"
        )
        plotted_crossenc_labels = [None] * len(points)
    else:
        validation_title = (
            f"{method.upper()} | token={record['token_text']!r} | sem_node={record['sem_node_id']} | "
            f"path={record['path_taken']}\n(cross-encoder per-point vote; ★ = FFT-sampled)"
        )
        plotted_crossenc_labels = crossenc_labels

    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=120)
    title_base = (
        f"{method.upper()} | token={record['token_text']!r} | "
        f"sem_node={record['sem_node_id']} | path={record['path_taken']}"
    )
    _scatter_by_label(
        axes[0], coords, recorded_labels, sampled_mask,
        f"{title_base}\n(recorded assignment; ★ = FFT-sampled)",
        all_candidate_labels=candidate_descriptions,
    )
    _scatter_by_label(
        axes[1], coords, plotted_crossenc_labels, sampled_mask,
        validation_title,
        all_candidate_labels=candidate_descriptions,
    )
    plt.tight_layout()

    saved_image_path = None
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight")
        saved_image_path = str(save_path)
    plt.close(fig)

    return {
        "record_index": record_index,
        "recorded_labels": recorded_labels,
        "crossenc_labels": crossenc_labels,
        "coords": coords,
        "candidate_descriptions": candidate_descriptions,
        "recorded_counts": dict(recorded_counts),
        "crossencoder_counts": None if crossenc_counts is None else dict(crossenc_counts),
        "count_deltas": None if count_deltas is None else dict(count_deltas),
        "crossencoder_available": crossenc_labels is not None,
        "image_path": saved_image_path,
    }


In [7]:
from datetime import datetime

VIS_EXPORT_ROOT = REPO_ROOT / "cache" / "fft_traversal_visualizations"
VIS_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
VIS_RUN_DIR = VIS_EXPORT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
VIS_IMAGE_DIR = VIS_RUN_DIR / "images"
VIS_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

VIS_REPORT_LINES = [
    f"fft_cache_path={current_fft_cache_path}",
    f"loaded_from_cache={loaded_from_cache}",
    f"live_graph_available={graph_database is not None}",
    "",
]

candidate_records = [r for r in fft_records if r["path_taken"] == "split" and len(r["all_text_embeddings"]) >= 12]
if not candidate_records:
    candidate_records = [r for r in fft_records if r["path_taken"] == "consensus" and len(r["all_text_embeddings"]) >= 12]
candidate_records.sort(key=lambda r: len(r["all_text_embeddings"]), reverse=True)

if not candidate_records:
    VIS_REPORT_LINES.extend([
        "method=pca",
        "status=no_records_with_minimum_point_threshold",
        "",
    ])
else:
    for selected_index, record in enumerate(candidate_records):
        image_name = f"{selected_index:02d}_{_safe_slug(record['token_text'])}_sem{record['sem_node_id']}_pca.png"
        export_result = visualize_fft_record(
            record,
            graph=graph_database,
            method="pca",
            save_path=VIS_IMAGE_DIR / image_name,
            record_index=selected_index,
        )
        if export_result is not None:
            VIS_REPORT_LINES.extend(_build_record_report_lines(record, export_result, method="pca"))


In [8]:
if "VIS_RUN_DIR" not in globals() or "VIS_IMAGE_DIR" not in globals():
    raise RuntimeError("Run the PCA export cell before the t-SNE export cell.")
if "VIS_REPORT_LINES" not in globals():
    VIS_REPORT_LINES = []
candidate_records = [r for r in fft_records if r["path_taken"] == "split" and len(r["all_text_embeddings"]) >= 12]
if not candidate_records:
    candidate_records = [r for r in fft_records if r["path_taken"] == "consensus" and len(r["all_text_embeddings"]) >= 12]
candidate_records.sort(key=lambda r: len(r["all_text_embeddings"]), reverse=True)

if not candidate_records:
    VIS_REPORT_LINES.extend([
        "method=tsne",
        "status=no_records_with_minimum_point_threshold",
        "",
    ])
else:
    for selected_index, record in enumerate(candidate_records):
        image_name = f"{selected_index:02d}_{_safe_slug(record['token_text'])}_sem{record['sem_node_id']}_tsne.png"
        export_result = visualize_fft_record(
            record,
            graph=graph_database,
            method="tsne",
            save_path=VIS_IMAGE_DIR / image_name,
            record_index=selected_index,
        )
        if export_result is not None:
            VIS_REPORT_LINES.extend(_build_record_report_lines(record, export_result, method="tsne"))

VIS_REPORT_PATH = VIS_RUN_DIR / "visualization_report.txt"
VIS_REPORT_PATH.write_text("\n".join(VIS_REPORT_LINES), encoding="utf-8")


27050

In [9]:
# Pick any record by index (see the dataframe above) and visualize.
# RECORD_IDX = 0
# visualize_fft_record(fft_records[RECORD_IDX], graph_database, method="pca")